# 01 — Data Audit & Cleaning

**Workflow:** FIND → DIAGNOSE → FIX → VALIDATE → DOCUMENT

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import sys
sys.path.append('../src')
from utils import detect_columns, clean_basic, add_common_features

In [5]:
raw = pd.read_csv("../data/Top Movies dataset.csv")

## 1. Dataset overview

In [6]:
display(pd.DataFrame({'column': raw.columns, 'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'missing_pct': (raw.isna().mean()*100).round(2), 'unique': raw.nunique(dropna=True)}).sort_values('missing_pct', ascending=False))

,column,dtype,missing,missing_pct,unique
Genre,Genre,object,11,0.11,2337
Poster_Url,Poster_Url,object,11,0.11,9826
Popularity,Popularity,float64,10,0.10,8160
Vote_Count,Vote_Count,object,10,0.10,3267
Vote_Average,Vote_Average,object,10,0.10,75
Original_Language,Original_Language,object,10,0.10,44
Title,Title,object,9,0.09,9514
Overview,Overview,object,9,0.09,9823
Release_Date,Release_Date,object,0,0.00,5903
Working URL,Working URL,object,0,0.00,9827


In [7]:
print('Duplicate rows:', raw.duplicated().sum())
print('Exact duplicate percentage:', round(raw.duplicated().mean()*100,2), '%')
display(raw.describe(include='all').T)

Duplicate rows: 0
Exact duplicate percentage: 0.0 %


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Release_Date,9837,5903,10/03/2022,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,9828,9514,Beauty and the Beast,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Overview,9828,9823,Dr. Raichi is one of the only survivors of the...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Popularity,9827.0,NaN,NaN,NaN,40.320672,108.875673,7.1,16.1275,21.191,35.1745,5083.954
Vote_Count,9827,3267,0,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vote_Average,9827,75,6.4,435,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Original_Language,9827,44,en,7569,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Genre,9826,2337,Drama,466,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Poster_Url,9826,9826,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Working URL,9837,9827,0,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Column detection and categorical audit

In [8]:
cols = detect_columns(raw)
cols

{'title': 'Title',
 'rating': 'Vote_Average',
 'popularity': 'Popularity',
 'votes': 'Vote_Count',
 'year': None,
 'release_date': 'Release_Date',
 'genre': 'Genre',
 'overview': 'Overview',
 'language': 'Original_Language'}

In [9]:
for c in raw.select_dtypes(include=['object','string','category']).columns:
    print(f'\n### {c} — {raw[c].nunique(dropna=True)} unique')
    display(raw[c].value_counts(dropna=False).head(15).to_frame('count'))


### Release_Date — 5903 unique


,count
Release_Date,
10/03/2022,16
09/03/2022,15
12/02/2021,13
08/10/2021,13
23/07/2021,11
01/10/2021,11
03/12/2021,10
12/11/2021,10
29/09/2017,9



### Title — 9514 unique


,count
Title,
NaN,9
Beauty and the Beast,4
Alice in Wonderland,4
The Little Mermaid,3
Awake,3
Carrie,3
Halloween,3
Les Misérables,3
Cinderella,3



### Overview — 9823 unique


...

,count
Genre,
Drama,466
Comedy,403
"Drama, Romance",248
Horror,238
"Horror, Thriller",199
"Comedy, Romance",196
Documentary,139
"Comedy, Drama",137
"Action, Thriller",125



### Poster_Url — 9826 unique


,count
Poster_Url,
NaN,11
https://image.tmdb.org/t/p/original/p3pHw85UMZPegfMZBA6dZ06yarm.jpg,1
https://image.tmdb.org/t/p/original/A3LEGz18fTXT1ot2TEtgFCPIOQY.jpg,1
https://image.tmdb.org/t/p/original/s3x7CZO7gCICKQkA5rSgorkJzYy.jpg,1
https://image.tmdb.org/t/p/original/8rMiBz8kLMNmQyMbQXL9MPIlyw.jpg,1
https://image.tmdb.org/t/p/original/31ykUoOS2RNdARk3B8z3y25gD8b.jpg,1
https://image.tmdb.org/t/p/original/8wm53EpaTLzQuJ3F19H5AwG3Fhl.jpg,1
https://image.tmdb.org/t/p/original/fLwbV87gyp5h0BmV7tNtnkmFimL.jpg,1
https://image.tmdb.org/t/p/original/deOzvJHnSSl8FI1HEJjPGgOsS9U.jpg,1



### Working URL — 9827 unique


,count
Working URL,
0,11
https://image.tmdb.org/t/p/original/p3pHw85UMZPegfMZBA6dZ06yarm.jpg,1
https://image.tmdb.org/t/p/original/A3LEGz18fTXT1ot2TEtgFCPIOQY.jpg,1
https://image.tmdb.org/t/p/original/s3x7CZO7gCICKQkA5rSgorkJzYy.jpg,1
https://image.tmdb.org/t/p/original/8rMiBz8kLMNmQyMbQXL9MPIlyw.jpg,1
https://image.tmdb.org/t/p/original/31ykUoOS2RNdARk3B8z3y25gD8b.jpg,1
https://image.tmdb.org/t/p/original/8wm53EpaTLzQuJ3F19H5AwG3Fhl.jpg,1
https://image.tmdb.org/t/p/original/fLwbV87gyp5h0BmV7tNtnkmFimL.jpg,1
https://image.tmdb.org/t/p/original/deOzvJHnSSl8FI1HEJjPGgOsS9U.jpg,1


## 3. Cleaning

In [10]:
df = clean_basic(raw)
df, cols = add_common_features(df)
print('After basic cleaning:', df.shape)
display(df.head())

After basic cleaning: (9837, 13)


c:\Users\Dell\Downloads\movie_analytics_project\movie_analytics_project\notebooks\../src\utils.py:47: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[cols['release_date']] = pd.to_datetime(df[cols['release_date']], errors='coerce')


,release_date,title,overview,popularity,vote_count,vote_average,original_language,genre,poster_url,working_url,release_year_derived,genre_count,decade
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...,https://image.tmdb.org/t/p/original/1g0dhYtq4i...,2021.0,3,2020
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...,https://image.tmdb.org/t/p/original/74xTEgt7R3...,2022.0,3,2020
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,https://image.tmdb.org/t/p/original/vDHsLnOWKl...,2022.0,1,2020
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...,https://image.tmdb.org/t/p/original/4j0PNHkMr5...,2021.0,4,2020
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...,2021.0,4,2020


In [11]:
# Numeric sanity checks for the detected core metrics
for label in ['rating','popularity','votes','year']:
    c = cols.get(label)
    if c and c in df.columns:
        print(label, c, 'min=', df[c].min(), 'max=', df[c].max(), 'missing=', df[c].isna().sum())

rating vote_average min= 0.0 max= 10.0 missing= 11
popularity popularity min= 7.1 max= 5083.954 missing= 10
votes vote_count min= 0 max= 31077 missing= 11
year release_year_derived min= 1902.0 max= 2024.0 missing= 10


## 4. Missing-value treatment

For analysis, do not blindly fill every column. Keep a documented decision: numeric summary/imputation only where appropriate; preserve unknown categorical values as `Unknown`; drop rows only for model-specific required fields.

In [12]:
for c in df.select_dtypes(include='object').columns:
    if df[c].isna().any():
        df[c] = df[c].fillna('Unknown')

print('Remaining missing values:')
display(df.isna().sum().sort_values(ascending=False).head(20))

Remaining missing values:


vote_count              11
vote_average            11
genre                   11
poster_url              11
release_date            10
popularity              10
original_language       10
release_year_derived    10
decade                  10
title                    9
overview                 9
working_url              0
genre_count              0
dtype: int64

## 5. Before vs after validation

In [13]:
validation = pd.DataFrame({
    'metric': ['rows','columns','duplicate_rows'],
    'before': [raw.shape[0], raw.shape[1], raw.duplicated().sum()],
    'after': [df.shape[0], df.shape[1], df.duplicated().sum()]
})
display(validation)

,metric,before,after
0,rows,9837,9837
1,columns,10,13
2,duplicate_rows,0,8


In [14]:
Path('../outputs').mkdir(exist_ok=True)
df.to_csv('../outputs/movies_cleaned.csv', index=False)
print('Saved:', Path('../outputs/movies_cleaned.csv').resolve())

Saved: C:\Users\Dell\Downloads\movie_analytics_project\movie_analytics_project\outputs\movies_cleaned.csv
